# 🎰 Multi-Armed Bandit Playground



## 📋 Table of Contents

1. [Introduction & Problem Framing](#1-introduction)
2. [Mathematical Foundations](#2-math)
3. [Environment Design](#3-environment)
4. [Algorithm Implementations](#4-algorithms)
5. [Experiment Framework](#5-framework)
6. [Visualization Suite](#6-visualizations)
7. [Core Experiments & Analysis](#7-experiments)
8. [Advanced Experiments](#8-advanced)
9. [Business Strategy Memo](#9-business)
10. [Interactive Explorer](#10-interactive)
11. [Utilities & Export](#11-utilities)
12. [Conclusions & Future Work](#12-conclusions)

---
## 1. Introduction & Problem Framing <a id="1-introduction"></a>

### The Problem

Imagine you're a product manager testing **5 different homepage designs**. Each design converts visitors at an unknown rate. You need to figure out which design is best — but every visitor shown a bad design is **lost revenue**.

This is the **multi-armed bandit problem**: balancing **exploration** (trying different options to learn) with **exploitation** (using what you already know to maximize reward).

### Why Not A/B Testing?

Traditional A/B tests split traffic equally across all variants for a fixed period, then pick the winner. This works, but:

- **Wastes traffic** on clearly inferior variants
- **Fixed duration** — can't adapt early
- **Binary outcome** — no gradual learning

Bandit algorithms **adapt in real-time**, shifting traffic toward better-performing variants while still exploring.

### What We'll Build

In this notebook, we simulate product experimentation scenarios and compare five bandit algorithms on:

- **Cumulative reward** — total conversions captured
- **Cumulative regret** — reward lost vs. always picking the best arm
- **Convergence speed** — how fast each algorithm finds the best option
- **Robustness** — performance under low traffic, many options, and changing environments

---
## 2. Mathematical Foundations <a id="2-math"></a>

### Regret

At each time step $t$, we choose arm $a_t$ and receive reward $r_t$. The **optimal arm** $a^*$ has the highest expected reward $\mu^*$. **Regret** measures the cost of not always picking the best:

$$R_T = \sum_{t=1}^{T} (\mu^* - \mu_{a_t})$$

Good algorithms achieve **sublinear regret** — regret grows slower than $T$, meaning we converge to the optimal arm.

### Key Algorithm Ideas

| Algorithm | Core Idea |
|---|---|
| **Epsilon-Greedy** | Exploit the best-known arm $(1-\varepsilon)$ of the time; explore randomly $\varepsilon$ of the time |
| **Optimistic Initial Values** | Initialize expected rewards high, so all arms get tried early as reality disappoints |
| **UCB1** | Pick the arm with highest $\bar{x}_a + c\sqrt{\frac{\ln t}{n_a}}$ — balances mean reward with uncertainty |
| **Thompson Sampling** | Sample from each arm's posterior $\text{Beta}(\alpha_a, \beta_a)$, pick the highest sample |
| **Sliding-Window TS** | Same as Thompson, but only uses the last $W$ observations — adapts to changing rewards |

In [ ]:
# ============================================================
# Imports & Configuration
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from collections import defaultdict
import warnings
import os

# Reproducibility
GLOBAL_SEED = 42

# Plot styling
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'figure.dpi': 100,
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'legend.fontsize': 10,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'lines.linewidth': 2,
})

# Color palette for algorithms
ALGO_COLORS = {
    'Epsilon-Greedy': '#E74C3C',
    'Optimistic Init': '#3498DB',
    'UCB1': '#2ECC71',
    'Thompson Sampling': '#9B59B6',
    'Sliding-Window TS': '#F39C12',
}

# Output directory for saved plots
OUTPUT_DIR = 'outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Setup complete. NumPy version:', np.__version__)

---
## 3. Environment Design <a id="3-environment"></a>

The `BanditEnvironment` simulates a product experiment with configurable arms (variants), reward probabilities, and optional non-stationary drift.

**Key features:**
- Configurable number of arms and reward probabilities
- Bernoulli reward distribution (click/no-click)
- Stationary or non-stationary (drifting) rewards
- Reproducible via random seeds

In [ ]:
# ============================================================
# BanditEnvironment
# ============================================================

class BanditEnvironment:
    """Simulates a multi-armed bandit product experiment.
    
    Parameters
    ----------
    reward_probs : list of float
        True reward probability for each arm (Bernoulli).
    stationary : bool
        If False, reward probabilities drift over time.
    drift_rate : float
        Std dev of Gaussian noise added to probs each step (non-stationary only).
    seed : int or None
        Random seed for reproducibility.
    """
    
    def __init__(self, reward_probs, stationary=True, drift_rate=0.001, seed=None):
        self.initial_probs = np.array(reward_probs, dtype=float)
        self.reward_probs = self.initial_probs.copy()
        self.n_arms = len(reward_probs)
        self.stationary = stationary
        self.drift_rate = drift_rate
        self.rng = np.random.RandomState(seed)
        self.step_count = 0
    
    def pull(self, arm):
        """Pull an arm and return a Bernoulli reward (0 or 1)."""
        reward = self.rng.random() < self.reward_probs[arm]
        self.step_count += 1
        if not self.stationary:
            self._drift()
        return float(reward)
    
    def _drift(self):
        """Apply bounded random walk to reward probabilities."""
        noise = self.rng.normal(0, self.drift_rate, self.n_arms)
        self.reward_probs = np.clip(self.reward_probs + noise, 0.01, 0.99)
    
    def get_optimal_arm(self):
        """Return the index of the current best arm."""
        return int(np.argmax(self.reward_probs))
    
    def get_optimal_reward(self):
        """Return the reward probability of the current best arm."""
        return float(np.max(self.reward_probs))
    
    def reset(self):
        """Reset environment to initial state."""
        self.reward_probs = self.initial_probs.copy()
        self.step_count = 0
    
    def __repr__(self):
        status = 'stationary' if self.stationary else f'non-stationary (drift={self.drift_rate})'
        return f'BanditEnvironment(arms={self.n_arms}, {status}, probs={self.reward_probs.round(3)})'


# Quick validation 
env = BanditEnvironment(reward_probs=[0.1, 0.3, 0.5, 0.7, 0.9], seed=42)
print(env)
print(f'Optimal arm: {env.get_optimal_arm()} (p={env.get_optimal_reward():.1f})')
print(f'Sample pulls from arm 4: {[env.pull(4) for _ in range(10)]}')

# Non-stationary test
env_ns = BanditEnvironment(reward_probs=[0.3, 0.5, 0.7], stationary=False, drift_rate=0.01, seed=42)
for _ in range(1000):
    env_ns.pull(0)
print(f'\nAfter 1000 steps with drift: {env_ns}')
print(f'Probs stayed in [0.01, 0.99]: {all(0.01 <= p <= 0.99 for p in env_ns.reward_probs)}')

---
## 4. Algorithm Implementations <a id="4-algorithms"></a>

All algorithms share a common `BanditAlgorithm` interface and track cumulative reward, cumulative regret, and arm selection counts.

| Class | Key Parameter | Strategy |
|---|---|---|
| `EpsilonGreedy` | `epsilon` | Random explore at rate ε |
| `OptimisticInitialValues` | `initial_value` | Greedy on inflated Q-values |
| `UCB1` | `c` | Upper confidence bound |
| `ThompsonSampling` | — | Beta posterior sampling |
| `SlidingWindowTS` | `window_size` | Windowed Beta posterior |

In [ ]:
# ============================================================
# Bandit Algorithms
# ============================================================

class BanditAlgorithm:
    """Base class for all bandit algorithms.
    
    Subclasses must implement select_arm() and update().
    """
    
    def __init__(self, n_arms, name='Base'):
        self.n_arms = n_arms
        self.name = name
        self.reset()
    
    def select_arm(self):
        raise NotImplementedError
    
    def update(self, arm, reward, optimal_reward=0.0):
        """Update state after observing a reward."""
        self.arm_counts[arm] += 1
        self.total_reward += reward
        self.total_regret += (optimal_reward - reward)
        self.reward_history.append(self.total_reward)
        self.regret_history.append(self.total_regret)
    
    def reset(self):
        """Reset all tracking state."""
        self.arm_counts = np.zeros(self.n_arms)
        self.total_reward = 0.0
        self.total_regret = 0.0
        self.reward_history = []
        self.regret_history = []
    
    def __repr__(self):
        return f'{self.name}(arms={self.n_arms})'


# 1. Epsilon-Greedy

class EpsilonGreedy(BanditAlgorithm):
    """Explores randomly with probability epsilon, exploits otherwise."""
    
    def __init__(self, n_arms, epsilon=0.1, seed=None):
        self.epsilon = epsilon
        self.q_values = np.zeros(n_arms)
        self.rng = np.random.RandomState(seed)
        super().__init__(n_arms, name='Epsilon-Greedy')
    
    def select_arm(self):
        if self.rng.random() < self.epsilon:
            return self.rng.randint(self.n_arms)
        return int(np.argmax(self.q_values))
    
    def update(self, arm, reward, optimal_reward=0.0):
        super().update(arm, reward, optimal_reward)
        # Incremental mean update
        n = self.arm_counts[arm]
        self.q_values[arm] += (reward - self.q_values[arm]) / n
    
    def reset(self):
        super().reset()
        self.q_values = np.zeros(self.n_arms)


# 2. Optimistic Initial Values

class OptimisticInitialValues(BanditAlgorithm):
    """Initializes Q-values high to encourage early exploration."""
    
    def __init__(self, n_arms, initial_value=5.0, seed=None):
        self.initial_value = initial_value
        self.q_values = np.full(n_arms, initial_value, dtype=float)
        self.rng = np.random.RandomState(seed)
        super().__init__(n_arms, name='Optimistic Init')
    
    def select_arm(self):
        # Break ties randomly
        max_q = np.max(self.q_values)
        candidates = np.where(self.q_values == max_q)[0]
        return int(self.rng.choice(candidates))
    
    def update(self, arm, reward, optimal_reward=0.0):
        super().update(arm, reward, optimal_reward)
        n = self.arm_counts[arm]
        self.q_values[arm] += (reward - self.q_values[arm]) / n
    
    def reset(self):
        super().reset()
        self.q_values = np.full(self.n_arms, self.initial_value, dtype=float)


# 3. UCB1 

class UCB1(BanditAlgorithm):
    """Upper Confidence Bound — balances exploitation with uncertainty bonus."""
    
    def __init__(self, n_arms, c=2.0, seed=None):
        self.c = c
        self.q_values = np.zeros(n_arms)
        self.rng = np.random.RandomState(seed)
        super().__init__(n_arms, name='UCB1')
    
    def select_arm(self):
        total_counts = np.sum(self.arm_counts)
        # Play each arm once first
        for arm in range(self.n_arms):
            if self.arm_counts[arm] == 0:
                return arm
        ucb_values = self.q_values + self.c * np.sqrt(np.log(total_counts) / self.arm_counts)
        return int(np.argmax(ucb_values))
    
    def update(self, arm, reward, optimal_reward=0.0):
        super().update(arm, reward, optimal_reward)
        n = self.arm_counts[arm]
        self.q_values[arm] += (reward - self.q_values[arm]) / n
    
    def reset(self):
        super().reset()
        self.q_values = np.zeros(self.n_arms)


# 4. Thompson Sampling (Beta-Bernoulli)

class ThompsonSampling(BanditAlgorithm):
    """Bayesian approach — samples from Beta posterior for each arm."""
    
    def __init__(self, n_arms, seed=None):
        self.alpha = np.ones(n_arms)  # successes + 1
        self.beta_param = np.ones(n_arms)   # failures + 1
        self.rng = np.random.RandomState(seed)
        super().__init__(n_arms, name='Thompson Sampling')
    
    def select_arm(self):
        samples = self.rng.beta(self.alpha, self.beta_param)
        return int(np.argmax(samples))
    
    def update(self, arm, reward, optimal_reward=0.0):
        super().update(arm, reward, optimal_reward)
        self.alpha[arm] += reward
        self.beta_param[arm] += (1 - reward)
    
    def reset(self):
        super().reset()
        self.alpha = np.ones(self.n_arms)
        self.beta_param = np.ones(self.n_arms)


# 5. Sliding-Window Thompson Sampling

class SlidingWindowTS(BanditAlgorithm):
    """Thompson Sampling using only the last W observations per arm.
    
    Adapts to non-stationary environments by forgetting old data.
    """
    
    def __init__(self, n_arms, window_size=200, seed=None):
        self.window_size = window_size
        self.windows = [[] for _ in range(n_arms)]
        self.rng = np.random.RandomState(seed)
        super().__init__(n_arms, name='Sliding-Window TS')
    
    def select_arm(self):
        samples = np.zeros(self.n_arms)
        for arm in range(self.n_arms):
            w = self.windows[arm][-self.window_size:] if self.windows[arm] else []
            alpha = sum(w) + 1
            beta_param = len(w) - sum(w) + 1
            samples[arm] = self.rng.beta(alpha, beta_param)
        return int(np.argmax(samples))
    
    def update(self, arm, reward, optimal_reward=0.0):
        super().update(arm, reward, optimal_reward)
        self.windows[arm].append(reward)
        # Keep only recent observations to save memory
        if len(self.windows[arm]) > self.window_size * 2:
            self.windows[arm] = self.windows[arm][-self.window_size:]
    
    def reset(self):
        super().reset()
        self.windows = [[] for _ in range(self.n_arms)]


# Quick validation
env_test = BanditEnvironment(reward_probs=[0.2, 0.5, 0.8], seed=0)
algos = [
    EpsilonGreedy(3, epsilon=0.1, seed=0),
    OptimisticInitialValues(3, initial_value=5.0, seed=0),
    UCB1(3, c=2.0, seed=0),
    ThompsonSampling(3, seed=0),
    SlidingWindowTS(3, window_size=200, seed=0),
]

for algo in algos:
    for _ in range(100):
        arm = algo.select_arm()
        reward = env_test.pull(arm)
        algo.update(arm, reward, env_test.get_optimal_reward())
    env_test.reset()
    print(f'{algo.name:20s} | reward={algo.total_reward:.0f} | regret={algo.total_regret:.1f} | counts={algo.arm_counts}')

---
## 5. Experiment Framework <a id="5-framework"></a>

The `ExperimentRunner` orchestrates simulations: it runs each algorithm multiple times with different seeds, averages the results, and computes convergence metrics.

**Tracked metrics (per step, averaged across runs):**
- Cumulative reward
- Cumulative regret
- Optimal arm selection %
- Convergence time (first step where optimal arm % > 90% is sustained)

In [ ]:
# ============================================================
# Experiment Framework
# ============================================================

# Configuration constants
DEFAULT_N_STEPS = 10000
DEFAULT_N_RUNS = 100
CONVERGENCE_THRESHOLD = 0.90  # 90% optimal arm selection
CONVERGENCE_WINDOW = 100      # sustained over this many steps


def create_algorithm(algo_class, n_arms, seed, **kwargs):
    """Factory to create an algorithm instance with a given seed."""
    return algo_class(n_arms=n_arms, seed=seed, **kwargs)


def run_single_simulation(algo, env, n_steps):
    """Run one simulation of an algorithm on an environment.
    
    Returns
    -------
    dict with keys: cumulative_reward, cumulative_regret, optimal_selections
        Each is a 1D array of length n_steps.
    """
    cumulative_reward = np.zeros(n_steps)
    cumulative_regret = np.zeros(n_steps)
    optimal_selections = np.zeros(n_steps)
    
    for t in range(n_steps):
        arm = algo.select_arm()
        optimal_arm = env.get_optimal_arm()
        optimal_reward = env.get_optimal_reward()
        reward = env.pull(arm)
        algo.update(arm, reward, optimal_reward)
        
        cumulative_reward[t] = algo.total_reward
        cumulative_regret[t] = algo.total_regret
        optimal_selections[t] = 1.0 if arm == optimal_arm else 0.0
    
    return {
        'cumulative_reward': cumulative_reward,
        'cumulative_regret': cumulative_regret,
        'optimal_selections': optimal_selections,
        'arm_counts': algo.arm_counts.copy(),
    }


def compute_convergence_time(optimal_pct, threshold=CONVERGENCE_THRESHOLD, window=CONVERGENCE_WINDOW):
    """Find the first step where optimal arm % exceeds threshold sustainably."""
    n = len(optimal_pct)
    if n < window:
        return n  # never converged
    for t in range(n - window):
        if np.all(optimal_pct[t:t + window] >= threshold):
            return t
    return n  # never converged


class ExperimentRunner:
    """Runs multiple simulations and aggregates results.
    
    Parameters
    ----------
    env_config : dict
        Arguments for BanditEnvironment constructor.
    algo_configs : list of (class, dict)
        Each entry is (AlgorithmClass, {keyword_args}).
    n_steps : int
        Time horizon per simulation.
    n_runs : int
        Number of independent runs to average.
    """
    
    def __init__(self, env_config, algo_configs, n_steps=DEFAULT_N_STEPS, n_runs=DEFAULT_N_RUNS):
        self.env_config = env_config
        self.algo_configs = algo_configs
        self.n_steps = n_steps
        self.n_runs = n_runs
    
    def run(self):
        """Run all simulations and return aggregated results.
        
        Returns
        -------
        dict : algorithm_name -> {
            'mean_reward': array, 'mean_regret': array,
            'mean_optimal_pct': array, 'mean_arm_counts': array,
            'convergence_time': int, 'final_reward': float,
            'final_regret': float
        }
        """
        results = {}
        n_arms = len(self.env_config['reward_probs'])
        
        for algo_class, algo_kwargs in self.algo_configs:
            # Accumulators
            all_reward = np.zeros((self.n_runs, self.n_steps))
            all_regret = np.zeros((self.n_runs, self.n_steps))
            all_optimal = np.zeros((self.n_runs, self.n_steps))
            all_arm_counts = np.zeros((self.n_runs, n_arms))
            
            for run_idx in range(self.n_runs):
                seed = GLOBAL_SEED + run_idx
                env = BanditEnvironment(**self.env_config, seed=seed)
                algo = create_algorithm(algo_class, n_arms, seed=seed + 1000, **algo_kwargs)
                
                sim = run_single_simulation(algo, env, self.n_steps)
                all_reward[run_idx] = sim['cumulative_reward']
                all_regret[run_idx] = sim['cumulative_regret']
                all_optimal[run_idx] = sim['optimal_selections']
                all_arm_counts[run_idx] = sim['arm_counts']
            
            # Running optimal % (cumulative average of binary selections)
            mean_optimal_pct = np.cumsum(all_optimal.mean(axis=0)) / np.arange(1, self.n_steps + 1)
            
            algo_name = create_algorithm(algo_class, n_arms, seed=0, **algo_kwargs).name
            results[algo_name] = {
                'mean_reward': all_reward.mean(axis=0),
                'mean_regret': all_regret.mean(axis=0),
                'mean_optimal_pct': mean_optimal_pct,
                'mean_arm_counts': all_arm_counts.mean(axis=0),
                'convergence_time': compute_convergence_time(mean_optimal_pct),
                'final_reward': all_reward[:, -1].mean(),
                'final_regret': all_regret[:, -1].mean(),
            }
        
        return results


# Quick validation
runner = ExperimentRunner(
    env_config={'reward_probs': [0.2, 0.5, 0.8]},
    algo_configs=[
        (EpsilonGreedy, {'epsilon': 0.1}),
        (ThompsonSampling, {}),
    ],
    n_steps=1000,
    n_runs=10,
)
test_results = runner.run()
for name, r in test_results.items():
    print(f"{name:20s} | final_reward={r['final_reward']:.1f} | final_regret={r['final_regret']:.1f} | convergence={r['convergence_time']}")

---
## 6. Visualization Suite <a id="6-visualizations"></a>

Reusable plotting functions for comparing algorithm performance. All plots use a consistent color palette and styling.

In [ ]:
# ============================================================
# Visualization Functions
# ============================================================

def _get_color(name):
    """Get algorithm color, with fallback."""
    return ALGO_COLORS.get(name, '#95A5A6')


def plot_cumulative_reward(results, title='Cumulative Reward', save=False):
    """Line plot of cumulative reward over time for each algorithm."""
    fig, ax = plt.subplots(figsize=(12, 6))
    for name, r in results.items():
        ax.plot(r['mean_reward'], label=name, color=_get_color(name))
    ax.set_xlabel('Step')
    ax.set_ylabel('Cumulative Reward')
    ax.set_title(title)
    ax.legend()
    if save:
        fig.savefig(f'{OUTPUT_DIR}/cumulative_reward.png', dpi=150, bbox_inches='tight')
    plt.show()


def plot_cumulative_regret(results, title='Cumulative Regret', save=False):
    """Line plot of cumulative regret over time for each algorithm."""
    fig, ax = plt.subplots(figsize=(12, 6))
    for name, r in results.items():
        ax.plot(r['mean_regret'], label=name, color=_get_color(name))
    ax.set_xlabel('Step')
    ax.set_ylabel('Cumulative Regret')
    ax.set_title(title)
    ax.legend()
    if save:
        fig.savefig(f'{OUTPUT_DIR}/cumulative_regret.png', dpi=150, bbox_inches='tight')
    plt.show()


def plot_arm_selection(results, env_config, title='Arm Selection Distribution', save=False):
    """Grouped bar chart showing how often each algorithm selected each arm."""
    n_algos = len(results)
    n_arms = len(env_config['reward_probs'])
    fig, ax = plt.subplots(figsize=(12, 6))
    
    bar_width = 0.8 / n_algos
    x = np.arange(n_arms)
    
    for i, (name, r) in enumerate(results.items()):
        counts = r['mean_arm_counts']
        pct = counts / counts.sum() * 100
        ax.bar(x + i * bar_width, pct, bar_width, label=name, color=_get_color(name), alpha=0.85)
    
    ax.set_xlabel('Arm')
    ax.set_ylabel('Selection %')
    ax.set_title(title)
    ax.set_xticks(x + bar_width * (n_algos - 1) / 2)
    ax.set_xticklabels([f'Arm {i}\n(p={env_config["reward_probs"][i]})' for i in range(n_arms)])
    ax.legend()
    if save:
        fig.savefig(f'{OUTPUT_DIR}/arm_selection.png', dpi=150, bbox_inches='tight')
    plt.show()


def plot_regret_growth_rate(results, window=500, title='Regret Growth Rate (Smoothed)', save=False):
    """Plot the derivative of regret to show how fast regret is growing."""
    fig, ax = plt.subplots(figsize=(12, 6))
    for name, r in results.items():
        regret = r['mean_regret']
        # Compute smoothed derivative
        rate = np.diff(regret)
        if len(rate) >= window:
            rate = np.convolve(rate, np.ones(window)/window, mode='valid')
        ax.plot(rate, label=name, color=_get_color(name))
    ax.set_xlabel('Step')
    ax.set_ylabel('Regret per Step')
    ax.set_title(title)
    ax.legend()
    if save:
        fig.savefig(f'{OUTPUT_DIR}/regret_growth_rate.png', dpi=150, bbox_inches='tight')
    plt.show()


def plot_convergence(results, title='Convergence: Optimal Arm Selection %', save=False):
    """Plot running optimal arm selection % over time."""
    fig, ax = plt.subplots(figsize=(12, 6))
    for name, r in results.items():
        ax.plot(r['mean_optimal_pct'] * 100, label=name, color=_get_color(name))
    ax.axhline(y=CONVERGENCE_THRESHOLD * 100, color='gray', linestyle='--', alpha=0.5, label=f'{CONVERGENCE_THRESHOLD*100:.0f}% threshold')
    ax.set_xlabel('Step')
    ax.set_ylabel('Optimal Arm Selection %')
    ax.set_title(title)
    ax.set_ylim(0, 105)
    ax.legend()
    if save:
        fig.savefig(f'{OUTPUT_DIR}/convergence.png', dpi=150, bbox_inches='tight')
    plt.show()


def plot_reward_distribution(env_config, title='True Arm Reward Probabilities'):
    """Bar chart of the true reward probabilities."""
    probs = env_config['reward_probs']
    fig, ax = plt.subplots(figsize=(8, 4))
    colors = ['#E74C3C' if p == max(probs) else '#3498DB' for p in probs]
    ax.bar(range(len(probs)), probs, color=colors, alpha=0.85)
    ax.set_xlabel('Arm')
    ax.set_ylabel('True Reward Probability')
    ax.set_title(title)
    ax.set_xticks(range(len(probs)))
    ax.set_ylim(0, 1.05)
    for i, p in enumerate(probs):
        ax.text(i, p + 0.02, f'{p:.2f}', ha='center', fontweight='bold')
    plt.show()


def print_summary_table(results):
    """Print a formatted summary table of experiment results."""
    print(f"{'Algorithm':<22} {'Final Reward':>14} {'Final Regret':>14} {'Convergence':>14} {'Best Arm %':>12}")
    print('-' * 78)
    for name, r in results.items():
        conv = r['convergence_time']
        conv_str = str(conv) if conv < len(r['mean_reward']) else 'Never'
        best_pct = r['mean_optimal_pct'][-1] * 100
        print(f"{name:<22} {r['final_reward']:>14.1f} {r['final_regret']:>14.1f} {conv_str:>14} {best_pct:>11.1f}%")


print('Visualization functions loaded.')

---
## 7. Core Experiments & Analysis <a id="7-experiments"></a>

### Experiment 1: Baseline Comparison

**Setup:** 5 arms with well-separated reward probabilities, 10,000 steps, averaged over 100 independent runs.

This is our primary benchmark — how do all five algorithms perform in a clean, stationary environment?

In [ ]:
# ============================================================
# Experiment 1: Baseline Comparison (5 arms, stationary)
# ============================================================

BASELINE_ENV = {
    'reward_probs': [0.1, 0.3, 0.5, 0.7, 0.9],
    'stationary': True,
}

BASELINE_ALGOS = [
    (EpsilonGreedy,          {'epsilon': 0.1}),
    (OptimisticInitialValues, {'initial_value': 5.0}),
    (UCB1,                   {'c': 2.0}),
    (ThompsonSampling,       {}),
    (SlidingWindowTS,        {'window_size': 200}),
]

# Show the environment
plot_reward_distribution(BASELINE_ENV, title='Experiment 1: True Arm Probabilities')

# Run the experiment
print('Running baseline experiment (10,000 steps × 100 runs)...')
runner_baseline = ExperimentRunner(
    env_config=BASELINE_ENV,
    algo_configs=BASELINE_ALGOS,
    n_steps=DEFAULT_N_STEPS,
    n_runs=DEFAULT_N_RUNS,
)
baseline_results = runner_baseline.run()
print('Done!\n')

# Summary table
print_summary_table(baseline_results)

# All plots
plot_cumulative_reward(baseline_results, title='Experiment 1: Cumulative Reward', save=True)
plot_cumulative_regret(baseline_results, title='Experiment 1: Cumulative Regret', save=True)
plot_arm_selection(baseline_results, BASELINE_ENV, title='Experiment 1: Arm Selection Distribution', save=True)
plot_regret_growth_rate(baseline_results, title='Experiment 1: Regret Growth Rate', save=True)
plot_convergence(baseline_results, title='Experiment 1: Convergence Speed', save=True)

### Baseline Results Analysis

**Key observations from the baseline experiment:**

1. **Thompson Sampling** typically achieves the lowest cumulative regret and fastest convergence - it efficiently balances exploration and exploitation via Bayesian reasoning.

2. **UCB1** performs well but may over-explore early due to its deterministic uncertainty bonus regret flattens later as confidence tightens.

3. **Epsilon-Greedy** has the most linear regret growth because it never stops exploring at rate ε, even after the optimal arm is known.

4. **Optimistic Initial Values** explores aggressively at the start (all arms look equally good), then converges. Performance depends heavily on the initial value chosen.

5. **Sliding-Window TS** performs similarly to Thompson Sampling in stationary environments but may be slightly worse due to discarding useful old data. Its strength appears in non-stationary settings (see Section 8).

The **regret growth rate** plot is particularly revealing - algorithms that converge show the rate approaching zero, while ε-greedy maintains a constant regret floor.

---
## 8. Advanced Experiments <a id="8-advanced"></a>

We now stress-test the algorithms under four challenging scenarios that reflect real product experimentation constraints.

### Experiment 2: Low Traffic vs High Traffic

Does the algorithm still work with only 1,000 visitors? How much does 50,000 help?

In [ ]:
# ============================================================
# Experiment 2: Traffic Volume Impact
# ============================================================

TRAFFIC_ENV = {'reward_probs': [0.1, 0.3, 0.5, 0.7, 0.9], 'stationary': True}
TRAFFIC_ALGOS = [
    (EpsilonGreedy,    {'epsilon': 0.1}),
    (UCB1,             {'c': 2.0}),
    (ThompsonSampling, {}),
]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, n_steps, label in zip(axes, [1000, 50000], ['Low Traffic (1K)', 'High Traffic (50K)']):
    runner = ExperimentRunner(env_config=TRAFFIC_ENV, algo_configs=TRAFFIC_ALGOS, n_steps=n_steps, n_runs=50)
    results = runner.run()
    for name, r in results.items():
        ax.plot(r['mean_regret'], label=name, color=_get_color(name))
    ax.set_title(label)
    ax.set_xlabel('Step')
    ax.set_ylabel('Cumulative Regret')
    ax.legend()

fig.suptitle('Experiment 2: Traffic Volume Impact on Regret', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(f'{OUTPUT_DIR}/exp2_traffic.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nKey insight: With low traffic, Thompson Sampling\'s Bayesian efficiency shines.')
print('Epsilon-Greedy wastes a larger fraction of limited visitors on random exploration.')

### Experiment 3: Few Arms (3) vs Many Arms (20)

More variants = harder exploration. How do algorithms scale?

In [ ]:
# ============================================================
# Experiment 3: Number of Arms
# ============================================================

few_probs = [0.3, 0.6, 0.9]
many_probs = list(np.linspace(0.05, 0.95, 20).round(3))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, probs, label in zip(axes, [few_probs, many_probs], ['3 Arms', '20 Arms']):
    env_cfg = {'reward_probs': probs, 'stationary': True}
    algo_cfgs = [
        (EpsilonGreedy,    {'epsilon': 0.1}),
        (UCB1,             {'c': 2.0}),
        (ThompsonSampling, {}),
    ]
    runner = ExperimentRunner(env_config=env_cfg, algo_configs=algo_cfgs, n_steps=10000, n_runs=50)
    results = runner.run()
    for name, r in results.items():
        ax.plot(r['mean_regret'], label=name, color=_get_color(name))
    ax.set_title(label)
    ax.set_xlabel('Step')
    ax.set_ylabel('Cumulative Regret')
    ax.legend()

fig.suptitle('Experiment 3: Scaling with Number of Arms', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(f'{OUTPUT_DIR}/exp3_arms.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nKey insight: UCB1 regret grows with more arms due to mandatory exploration.')
print('Thompson Sampling scales more gracefully — it quickly dismisses low-probability arms.')

### Experiment 4: Close vs Widely Separated Rewards

When all arms have similar probabilities (~0.5), distinguishing the best arm is much harder.

In [ ]:
# ============================================================
# Experiment 4: Reward Separation
# ============================================================

close_probs   = [0.45, 0.47, 0.49, 0.51, 0.53]
wide_probs    = [0.1, 0.3, 0.5, 0.7, 0.9]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, probs, label in zip(axes, [close_probs, wide_probs], ['Close (0.45\u20130.53)', 'Wide (0.1\u20130.9)']):
    env_cfg = {'reward_probs': probs, 'stationary': True}
    algo_cfgs = [
        (EpsilonGreedy,    {'epsilon': 0.1}),
        (UCB1,             {'c': 2.0}),
        (ThompsonSampling, {}),
    ]
    runner = ExperimentRunner(env_config=env_cfg, algo_configs=algo_cfgs, n_steps=10000, n_runs=50)
    results = runner.run()
    for name, r in results.items():
        ax.plot(r['mean_regret'], label=name, color=_get_color(name))
    ax.set_title(label)
    ax.set_xlabel('Step')
    ax.set_ylabel('Cumulative Regret')
    ax.legend()

fig.suptitle('Experiment 4: Impact of Reward Separation', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(f'{OUTPUT_DIR}/exp4_separation.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nKey insight: Close distributions make ALL algorithms struggle \u2014 the cost of misidentification is low,')
print('but convergence is slow. Thompson Sampling still edges ahead due to its probabilistic narrowing.')

### Experiment 5: Non-Stationary Environment

Reward probabilities drift over time. This is where Sliding-Window Thompson Sampling should shine \u2014 it forgets old data and adapts.

In [ ]:
# ============================================================
# Experiment 5: Non-Stationary Drift
# ============================================================

DRIFT_ENV = {
    'reward_probs': [0.3, 0.5, 0.7],
    'stationary': False,
    'drift_rate': 0.005,
}

DRIFT_ALGOS = [
    (EpsilonGreedy,    {'epsilon': 0.1}),
    (UCB1,             {'c': 2.0}),
    (ThompsonSampling, {}),
    (SlidingWindowTS,  {'window_size': 200}),
]

runner_drift = ExperimentRunner(
    env_config=DRIFT_ENV,
    algo_configs=DRIFT_ALGOS,
    n_steps=20000,
    n_runs=50,
)

print('Running non-stationary experiment (20,000 steps \u00d7 50 runs)...')
drift_results = runner_drift.run()
print('Done!\n')

print_summary_table(drift_results)
plot_cumulative_regret(drift_results, title='Experiment 5: Regret Under Non-Stationary Drift', save=True)
plot_regret_growth_rate(drift_results, window=1000, title='Experiment 5: Regret Growth Under Drift', save=True)

print('\nKey insight: Standard Thompson Sampling locks onto an arm and cannot adapt when rewards shift.')
print('Sliding-Window TS forgets old data and tracks the changing optimum.')
print('Epsilon-Greedy\'s constant exploration provides some robustness to drift.')

---
## 9. Business Strategy Memo <a id="9-business"></a>

*TBD

---
## 10. Interactive Explorer <a id="10-interactive"></a>

*Coming in Step 10*

In [ ]:
# Interactive widgets

---
## 11. Utilities & Export <a id="11-utilities"></a>

*Coming in Step 10*

In [ ]:
# Export utilities

---
## 12. Conclusions & Future Work <a id="12-conclusions"></a>
